# Assignment 2 - Recommender Systems
**Anime domain (MyAnimeList)**

We build a **Multi-Modal Two-Tower** recommendation model in PyTorch using the MyAnimeList dataset.

The item tower combines three feature streams via **learned attention fusion** (each modality votes on how much it matters) instead of a plain concat. The text stream uses **TF-IDF + SVD** — classic linear algebra, no transformers needed.

---
- **User Tower**: user_id embedding + user's average rating → small MLP
- **Item Tower**: 3 separate branches (categorical / numerical / text) fused with learned weights
- **Loss**: InfoNCE (in-batch negatives — same idea as YouTube DNN)
- **Eval**: Recall@K and NDCG@K

---
## 0. Install & Imports

In [ ]:
import os, math, random, zipfile, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

---
## 1. Load Data

Dataset: [MyAnimeList Dataset](https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database)  
Download the zip from Kaggle and place `anime.csv` and `rating.csv` in the same folder as this notebook.

In [ ]:
anime_df   = pd.read_csv('anime.csv')
ratings_df = pd.read_csv('rating.csv')

print(f'Anime entries : {len(anime_df):,}')
print(f'Rating entries: {len(ratings_df):,}')
anime_df.head(3)

---
## 2. Data Preprocessing

### 2a. Clean anime metadata

In [ ]:
# Drop rows missing the key fields we need
anime_df = anime_df.dropna(subset=['genre', 'type', 'rating']).reset_index(drop=True)

# --- Categorical features ---
# 'type': TV, Movie, OVA, ONA, Special, Music  →  encode as integer
# 'genre': multi-label string like "Action, Adventure" →  we just take the *first* genre as the primary one
anime_df['primary_genre'] = anime_df['genre'].str.split(',').str[0].str.strip()

type_enc  = LabelEncoder().fit(anime_df['type'])
genre_enc = LabelEncoder().fit(anime_df['primary_genre'])

anime_df['type_idx']  = type_enc.transform(anime_df['type'])
anime_df['genre_idx'] = genre_enc.transform(anime_df['primary_genre'])

# --- Numerical features ---
# episodes: fill missing with median, then log-scale so huge values don't dominate
anime_df['episodes'] = pd.to_numeric(anime_df['episodes'], errors='coerce')
anime_df['episodes'] = anime_df['episodes'].fillna(anime_df['episodes'].median())
anime_df['episodes_log'] = np.log1p(anime_df['episodes'])

# rating: the anime's community score (1-10) — normalise to [0,1]
anime_df['rating_norm'] = anime_df['rating'] / 10.0

# members: how many people added it to their list — log-scale
anime_df['members_log'] = np.log1p(anime_df['members'])

# Stack the 3 numerical features into one column for convenience
num_cols = ['episodes_log', 'rating_norm', 'members_log']

# Min-max normalise each numerical column to [0,1]
for c in num_cols:
    mn, mx = anime_df[c].min(), anime_df[c].max()
    anime_df[c] = (anime_df[c] - mn) / (mx - mn + 1e-9)

# --- Text feature ---
# 'name': the anime title.  We use it as a lightweight text signal.
# In practice you'd want a synopsis, but MAL dataset doesn't include one.
# TF-IDF on the title still captures word patterns (e.g. "Dragon", "No Game").
anime_df['text'] = anime_df['name'].fillna('').str.lower()

print(f'Clean anime entries: {len(anime_df):,}')
print(f'Type categories  : {type_enc.classes_}')
print(f'Genre categories : {len(genre_enc.classes_)}')

### 2b. Build TF-IDF text embeddings (SVD-compressed)

In [ ]:
TEXT_DIM = 32  # compress TF-IDF down to 32 dimensions via SVD

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(anime_df['text'])   # sparse (N_anime, 5000)

svd = TruncatedSVD(n_components=TEXT_DIM, random_state=SEED)
X_text = svd.fit_transform(X_tfidf).astype(np.float32)  # dense (N_anime, 32)

print(f'Text matrix shape: {X_text.shape}')
print(f'SVD explained variance: {svd.explained_variance_ratio_.sum():.2%}')

### 2c. Clean ratings & build interaction table

In [ ]:
# -1 in the rating column means the user watched it but didn't rate it
# We keep explicit ratings only (1-10)
ratings_df = ratings_df[ratings_df['rating'] != -1].copy()

# Keep only anime_ids that survived our metadata cleaning
valid_ids = set(anime_df['anime_id'].values)
ratings_df = ratings_df[ratings_df['anime_id'].isin(valid_ids)].reset_index(drop=True)

# To keep training fast: sample users that have rated at least 20 anime
counts = ratings_df.groupby('user_id').size()
active_users = counts[counts >= 20].index
ratings_df = ratings_df[ratings_df['user_id'].isin(active_users)].reset_index(drop=True)

# Re-index users and anime to contiguous integers
users_all  = sorted(ratings_df['user_id'].unique())
anime_all  = sorted(ratings_df['anime_id'].unique())

u2i = {u: i for i, u in enumerate(users_all)}
a2i = {a: i for i, a in enumerate(anime_all)}
i2a = {i: a for a, i in a2i.items()}

N_USERS = len(users_all)
N_ANIME = len(anime_all)

ratings_df['uid'] = ratings_df['user_id'].map(u2i)
ratings_df['aid'] = ratings_df['anime_id'].map(a2i)

# User's mean rating — used as an extra signal in the user tower
user_mean_rating = ratings_df.groupby('uid')['rating'].mean() / 10.0  # normalised
user_mean_tensor = torch.zeros(N_USERS)
for uid, val in user_mean_rating.items():
    user_mean_tensor[uid] = float(val)
user_mean_tensor = user_mean_tensor.to(device)

print(f'Users: {N_USERS:,}  |  Anime: {N_ANIME:,}  |  Interactions: {len(ratings_df):,}')

### 2d. Align item feature tensors to the re-indexed anime IDs

In [ ]:
# Build a lookup from anime_id → row in anime_df
anime_df = anime_df.set_index('anime_id')

# For each re-indexed anime, pull its features in order
item_type  = torch.zeros(N_ANIME, dtype=torch.long)
item_genre = torch.zeros(N_ANIME, dtype=torch.long)
item_num   = torch.zeros(N_ANIME, 3)       # [episodes_log, rating_norm, members_log]
item_text  = torch.zeros(N_ANIME, TEXT_DIM)

for aid, idx in a2i.items():
    if aid not in anime_df.index:
        continue
    row = anime_df.loc[aid]
    item_type[idx]  = int(row['type_idx'])
    item_genre[idx] = int(row['genre_idx'])
    item_num[idx]   = torch.tensor([row['episodes_log'], row['rating_norm'], row['members_log']])
    # text: find the original position in anime_df (before set_index it was row-aligned with X_text)
    orig_pos = anime_df.index.get_loc(aid)
    item_text[idx]  = torch.from_numpy(X_text[orig_pos])

item_type  = item_type.to(device)
item_genre = item_genre.to(device)
item_num   = item_num.to(device)
item_text  = item_text.to(device)

N_TYPES  = int(item_type.max().item()) + 1
N_GENRES = int(item_genre.max().item()) + 1
print(f'Item tensors ready. Types: {N_TYPES}  Genres: {N_GENRES}')

### 2e. Train / Test split

In [ ]:
# For each user, hold out their most recent interaction as the test item.
# Everything else goes to training. This is the "leave-one-out" protocol
# common in recommendation papers.
ratings_df = ratings_df.sort_values(['uid', 'rating'], ascending=[True, False])

test_rows  = ratings_df.groupby('uid').first().reset_index()   # best-rated per user = test
test_set   = dict(zip(test_rows['uid'], test_rows['aid']))     # uid -> aid

# Remove test interactions from the training set
test_idx   = ratings_df.groupby('uid')['aid'].transform('first') == ratings_df['aid']
train_df   = ratings_df[~test_idx].reset_index(drop=True)

# Pre-build per-user positive sets (needed for negative sampling)
user_pos = {}
for row in train_df.itertuples():
    user_pos.setdefault(row.uid, set()).add(row.aid)

print(f'Train: {len(train_df):,}  |  Test users: {len(test_set):,}')

---
## 3. Model Architecture

### The Two Towers

```
USER TOWER
  user_id  →  Embedding(DIM)
  user_mean_rating  →  scalar
  concat  →  MLP  →  DIM-dim vector

ITEM TOWER  (late fusion with learned weights)
  type + primary_genre  →  Embeddings  →  MLP  →  DIM   ← categorical branch
  episodes, score, members  →  MLP  →  DIM                ← numerical branch
  TF-IDF SVD (32d)  →  MLP  →  DIM                        ← text branch
  weighted_sum([cat, num, txt], softmax(w))  →  DIM-dim vector

Score = dot(user_vec, item_vec)
```

The fusion weights `w` are **learned** — the model figures out which branch is most useful.

In [ ]:
from tqdm.auto import tqdm

header = f"{'Epoch':>6}  {'Loss':>8}  {'Recall@10':>10}  {'Recall@20':>10}  {'Recall@50':>10}  {'NDCG@10':>8}  {'NDCG@20':>8}  {'NDCG@50':>8}"
print(header)
print('-' * len(header))

for ep in range(1, EPOCHS + 1):
    model.train()
    total_loss, steps = 0.0, 0

    loop = tqdm(train_loader, desc=f'Epoch {ep}/{EPOCHS}', leave=False)
    for uid, pos, neg in loop:
        uid = uid.to(device); pos = pos.to(device); neg = neg.to(device)

        u, p, n = model(uid, pos, neg)
        loss    = infonce_loss(u, p, n)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()
        steps += 1
        loop.set_postfix(loss=f'{total_loss/steps:.4f}')

    rec, ndc = evaluate()
    print(f"{'Epoch '+str(ep):>6}  {total_loss/steps:>8.4f}  "
          f"{rec[10]:>10.2%}  {rec[20]:>10.2%}  {rec[50]:>10.2%}  "
          f"{ndc[10]:>8.2%}  {ndc[20]:>8.2%}  {ndc[50]:>8.2%}")

# Final summary
w = F.softmax(model.item_tower.fusion_w, dim=0).detach().cpu().numpy()
rec, ndc = evaluate()
print(f"""
Final Results
-------------
Recall@10  :  {rec[10]:.2%}
Recall@20  :  {rec[20]:.2%}
Recall@50  :  {rec[50]:.2%}
NDCG@10    :  {ndc[10]:.2%}
NDCG@20    :  {ndc[20]:.2%}
NDCG@50    :  {ndc[50]:.2%}

Learned Fusion Weights
Categorical : {w[0]:.3f}
Numerical   : {w[1]:.3f}
Text        : {w[2]:.3f}
""")

---
## 4. Training

### InfoNCE loss (in-batch negatives)

For a batch of B (user, anime) pairs, each user's positive anime competes against all **other** anime in the same batch. The loss is a cross-entropy over B classes — the correct class being the diagonal. This is the same loss used in CLIP and Google's Two-Tower paper.

In [ ]:
# Build a reverse lookup: anime_id → name (for display)
id2name = anime_df['name'].to_dict()   # anime_df is indexed by anime_id at this point


@torch.no_grad()
def recommend(user_idx, k=10, exclude_seen=True):
    model.eval()

    uid_t  = torch.tensor([user_idx], device=device)
    u_vec  = model.user_vec(uid_t)                        # (1, DIM)

    all_ids  = torch.arange(N_ANIME, device=device)
    all_vecs = model.item_vec(all_ids)                    # (N_ANIME, DIM)
    scores   = (u_vec @ all_vecs.T).squeeze(0).cpu()     # (N_ANIME,)

    if exclude_seen:
        for aid in user_pos.get(user_idx, []):
            scores[aid] = -1e9

    top_k = scores.argsort(descending=True)[:k].tolist()
    return [(id2name.get(i2a[aid], f'id:{i2a[aid]}'), scores[aid].item()) for aid in top_k]


# Show top 10 recommendations for 5 different users
for uid in range(5):
    print(f'\nTop 10 for user {uid}:')
    for i, (name, score) in enumerate(recommend(uid, k=10), 1):
        print(f'  {i:>2}. {score:+.3f}  {name}')

In [ ]:
@torch.no_grad()
def evaluate(ks=(10, 20, 50)):
    model.eval()

    # Pre-compute all item vectors once
    all_ids  = torch.arange(N_ANIME, device=device)
    all_vecs = model.item_vec(all_ids)          # (N_ANIME, DIM)

    recall = {k: [] for k in ks}
    ndcg   = {k: [] for k in ks}

    test_uids = list(test_set.keys())

    for start in range(0, len(test_uids), 256):
        batch_uids = test_uids[start:start + 256]
        uid_t      = torch.tensor(batch_uids, device=device)
        u_vecs     = model.user_vec(uid_t)           # (chunk, DIM)
        scores     = u_vecs @ all_vecs.T             # (chunk, N_ANIME)

        for i, uid in enumerate(batch_uids):
            true_aid = test_set[uid]
            ranked   = scores[i].argsort(descending=True).cpu().tolist()

            for k in ks:
                top_k = ranked[:k]
                hit   = int(true_aid in top_k)
                recall[k].append(hit)
                # NDCG: 1/log2(rank+2) if found, else 0
                if hit:
                    rank = top_k.index(true_aid)
                    ndcg[k].append(1.0 / math.log2(rank + 2))
                else:
                    ndcg[k].append(0.0)

    return (
        {k: float(np.mean(recall[k])) for k in ks},
        {k: float(np.mean(ndcg[k]))   for k in ks},
    )

In [ ]:
header = f"{'Ep':>3}  {'Loss':>8}  {'R@10':>7}  {'R@20':>7}  {'R@50':>7}  {'N@10':>7}  {'N@20':>7}  {'N@50':>7}"
print(header)
print('-' * len(header))

for ep in range(1, EPOCHS + 1):
    model.train()
    total_loss, steps = 0.0, 0

    for uid, pos, neg in train_loader:
        uid = uid.to(device); pos = pos.to(device); neg = neg.to(device)

        u, p, n = model(uid, pos, neg)
        loss    = infonce_loss(u, p, n)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()
        steps += 1

    rec, ndc = evaluate()
    print(f"{ep:>3}  {total_loss/steps:>8.4f}  "
          f"{rec[10]:>7.4f}  {rec[20]:>7.4f}  {rec[50]:>7.4f}  "
          f"{ndc[10]:>7.4f}  {ndc[20]:>7.4f}  {ndc[50]:>7.4f}")

# Show the learned fusion weights after training
w = F.softmax(model.item_tower.fusion_w, dim=0).detach().cpu().numpy()
print(f'\nLearned fusion weights — Categorical: {w[0]:.3f} | Numerical: {w[1]:.3f} | Text: {w[2]:.3f}')

---
## 5. Make Recommendations

In [ ]:
# Build a reverse lookup: anime_id → name (for display)
id2name = anime_df['name'].to_dict()   # anime_df is indexed by anime_id at this point


@torch.no_grad()
def recommend(user_idx, k=10, exclude_seen=True):
    model.eval()

    uid_t  = torch.tensor([user_idx], device=device)
    u_vec  = model.user_vec(uid_t)                        # (1, DIM)

    all_ids  = torch.arange(N_ANIME, device=device)
    all_vecs = model.item_vec(all_ids)                    # (N_ANIME, DIM)
    scores   = (u_vec @ all_vecs.T).squeeze(0).cpu()     # (N_ANIME,)

    if exclude_seen:
        for aid in user_pos.get(user_idx, []):
            scores[aid] = -1e9

    top_k = scores.argsort(descending=True)[:k].tolist()
    return [(id2name.get(i2a[aid], f'id:{i2a[aid]}'), scores[aid].item()) for aid in top_k]


# Demo: pick a few users and show their top-5 recommendations
for uid in [0, 1, 2]:
    print(f'\nTop 5 for user {uid}:')
    for name, score in recommend(uid, k=5):
        print(f'  {score:+.3f}  {name}')